Imports

In [6]:
import pandas as pd

Load Data

In [7]:
finviz_data = pd.read_csv('Data/finviz_news.csv')
finnhub_data = pd.read_csv('Data/finnhub_news.csv')
gsib_data = pd.read_csv('Data/GSIB_Sample_News.csv')

Inspect Data

In [8]:
finviz_data.head()
finnhub_data.head()
gsib_data.head()

,date,ticker_symbol,title,link,sentiment
0,2026-04-09T21:11:04+00:00,JPM,How The Banco Del Bajío (BMV:BBAJIO O) Valuati...,https://finance.yahoo.com/markets/stocks/artic...,"{'polarity': 0.996, 'neg': 0.032, 'neu': 0.85,..."
1,2026-04-09T21:05:10+00:00,JPM,JPMorgan Chase &amp; Co. (JPM) – Among the 13 ...,https://finance.yahoo.com/markets/stocks/artic...,"{'polarity': 0.988, 'neg': 0.048, 'neu': 0.823..."
2,2026-04-09T19:05:16+00:00,JPM,Analyst Upgrades This Bank As Banking Stocks E...,https://finance.yahoo.com/m/db7590aa-50eb-3474...,"{'polarity': 0.823, 'neg': 0, 'neu': 0.664, 'p..."
3,2026-04-09T18:55:00+00:00,JPM,Merrill Recruits Advisors Who Oversaw $2.8 Bil...,https://finance.yahoo.com/m/17647796-b073-3278...,"{'polarity': 0.599, 'neg': 0, 'neu': 0.898, 'p..."
4,2026-04-09T16:04:22+00:00,JPM,"Home Depot CFO Warns Demand Softening, Housing...",https://finance.yahoo.com/economy/articles/hom...,"{'polarity': 0.998, 'neg': 0.042, 'neu': 0.851..."


Clean Data

In [9]:
import re
from ast import literal_eval

def parse_finviz_cell(raw_value):
    if pd.isna(raw_value):
        return pd.Series({'date': pd.NaT, 'headline': pd.NA})
    text = str(raw_value).strip()
    try:
        parsed = literal_eval(text)
        if isinstance(parsed, tuple) and len(parsed) >= 2:
            raw_date, raw_headline = parsed[0], parsed[1]
            return pd.Series({
                'date': pd.to_datetime(raw_date, errors='coerce'),
                'headline': str(raw_headline).strip() if raw_headline is not None else pd.NA
            })
    except Exception:
        pass
    date_match = re.search(r"\(\s*'([^']+)'\s*,", text)
    head_match = re.search(r"\(\s*'[^']+'\s*,\s*'(.+?)'\s*,\s*'https?://", text)
    fallback_date = pd.to_datetime(date_match.group(1), errors='coerce') if date_match else pd.NaT
    fallback_headline = head_match.group(1).strip() if head_match else pd.NA
    return pd.Series({'date': fallback_date, 'headline': fallback_headline})

finviz_long = finviz_data.melt(var_name='stock', value_name='raw_news').dropna(subset=['raw_news'])
parsed = finviz_long['raw_news'].apply(parse_finviz_cell)
finviz_clean = pd.concat([finviz_long[['stock']], parsed], axis=1)

finnhub_clean = finnhub_data.rename(columns={'ticker_symbol': 'stock'})[['stock', 'headline', 'date']].copy()
finnhub_clean['date'] = pd.to_datetime(finnhub_clean['date'], errors='coerce')
finnhub_clean['headline'] = finnhub_clean['headline'].astype(str).str.strip()

# GSIB data standardization (different column names: ticker_symbol, title)
gsib_clean = gsib_data.rename(columns={'ticker_symbol': 'stock', 'title': 'headline'})[['stock', 'headline', 'date']].copy()
gsib_clean['date'] = pd.to_datetime(gsib_clean['date'], errors='coerce')
gsib_clean['headline'] = gsib_clean['headline'].astype(str).str.strip()

for df in (finviz_clean, finnhub_clean, gsib_clean):
    df['stock'] = df['stock'].astype(str).str.upper().str.strip()
    df['headline'] = df['headline'].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

finviz_clean = finviz_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finviz_clean['date_only'] = finviz_clean['date'].dt.date
finnhub_clean = finnhub_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finnhub_clean['date_only'] = finnhub_clean['date'].dt.date
gsib_clean = gsib_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
gsib_clean['date_only'] = gsib_clean['date'].dt.date

Save Cleaned Data

In [10]:
finviz_clean.to_csv('Data/finviz_clean.csv', index=False)
finnhub_clean.to_csv('Data/finnhub_clean.csv', index=False)
gsib_clean.to_csv('Data/GSIB_clean.csv', index=False)